# Molecule Properties Prediction

# Deep Learning

## 1. GCN

## 2. D-MPNN

In [65]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import pandas as pd
import chemprop
from chemprop import data, featurizers
import torch
import torch.nn as nn
from model.dmpnn_focal import MPNNModel_FocalLoss
from model.utils import compute_classification_report

In [67]:
tox21_tasks = ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
               'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


tox21_data = pd.read_csv('../data/tox21.csv')
smiles = tox21_data.loc[:, 'smiles'].values
targets = tox21_data.loc[:, tox21_tasks].values
num_workers = 0

# Convert to Chemprop's MoleculeDatapoint format
all_data = [data.MoleculeDatapoint.from_smi(smile, target) for smile, target in zip(smiles, targets)]

# Transform into RDkit Mol objects for structure based splits
mols = [data.mol for data in all_data]
train_indices, val_indices, test_indices = data.make_split_indices(mols, "random", (0.7, 0.2, 0.1))
train_data, val_data, test_data = data.split_data_by_indices(
    all_data, train_indices, val_indices, test_indices
)


# Featurize the data
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

train_data = data.MoleculeDataset(train_data[0], featurizer)
val_data = data.MoleculeDataset(val_data[0], featurizer)
test_data = data.MoleculeDataset(test_data[0], featurizer)

# Create dataloaders
train_loader = data.build_dataloader(train_data, num_workers=num_workers)
val_loader = data.build_dataloader(val_data, num_workers=num_workers, shuffle=False)
test_loader = data.build_dataloader(test_data, num_workers=num_workers, shuffle=False)

[20:43:59] WARNING: not removing hydrogen atom without neighbors
The return type of make_split_indices has changed in v2.1 - see help(make_split_indices)
Dropping last batch of size 1 to avoid issues with batch normalization (dataset size = 1601, batch_size = 64)


### 1) Initial

In [ ]:
from lightning import pytorch as pl

mp = chemprop.nn.BondMessagePassing()
agg = chemprop.nn.MeanAggregation()
ffn = chemprop.nn.BinaryClassificationFFN(n_tasks=len(tox21_tasks))

batch_norm = False
metric_list = None   # AUROC used by default
mpnn = chemprop.models.MPNN(mp, agg, ffn, batch_norm, metric_list)

# trainer_mpnn = pl.Trainer(
#     logger=False,
#     enable_checkpointing=False, # Use `True` if you want to save model checkpoints. The checkpoints will be saved in the `checkpoints` folder.
#     enable_progress_bar=True,
#     accelerator="cpu",
#     devices=1,
#     max_epochs=20, # number of epochs to train for
# )

# trainer_mpnn.fit(mpnn, train_loader)


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
Loading `train_dataloader` to estimate number of stepping batches.
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

  | Name            | Type                    | Params | Mode 
----------------------------------------------

Epoch 19: 100%|██████████| 88/88 [00:03<00:00, 23.95it/s, train_loss_step=0.182, train_loss_epoch=0.178]

`Trainer.fit` stopped: `max_epochs=20` reached.


Epoch 19: 100%|██████████| 88/88 [00:03<00:00, 23.94it/s, train_loss_step=0.182, train_loss_epoch=0.178]


#### · Performancce on Validation Set

In [87]:
trainer_mpnn = pl.Trainer(logger=True)  # default TensorBoard logger
trainer_mpnn.validate(model=mpnn, dataloaders=val_loader)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Validation DataLoader 0: 100%|██████████| 25/25 [00:01<00:00, 13.68it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          val/roc          │    0.8699097633361816     │
│         val_loss          │    0.18786557018756866    │
└───────────────────────────┴───────────────────────────┘

[{'val/roc': 0.8699097633361816, 'val_loss': 0.18786557018756866}]

In [88]:
import torch

pred_probs = trainer_mpnn.predict(mpnn, dataloaders=val_loader)[:-1]
probs = torch.cat(pred_probs, dim=0).cpu().numpy()

all_y = []
for batch in val_loader:
    y = batch.Y  # 👈 get the label tensor from TrainingBatch
    all_y.append(y)
y_true = torch.cat(all_y, dim=0).cpu().numpy()  # shape: (num_samples, num_tasks)

model0_report_val = compute_classification_report(y_true, probs, threshold=0.6)
model0_report_val


/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 26/26 [00:00<00:00, 29.40it/s]
 - Precision: 0.5894
 - Recall: 0.1666
 - F1: 0.2426
 - AUC: 0.8406


,task,precision,recall,f1,auc,positives,predicted_positives,total_num
0,task_0,0.928571,0.433333,0.590909,0.823057,60,28,1501
1,task_1,0.888889,0.400000,0.551724,0.868053,60,27,1400
2,task_2,0.777778,0.302469,0.435556,0.910975,162,63,1349
3,task_3,0.000000,0.000000,0.000000,0.865150,56,0,1200
4,task_4,0.791667,0.111765,0.195876,0.739373,170,24,1269
5,task_5,0.833333,0.125000,0.217391,0.818838,80,12,1429
6,task_6,0.000000,0.000000,0.000000,0.822581,40,0,1342
7,task_7,0.775510,0.200000,0.317992,0.828900,190,49,1184
8,task_8,0.000000,0.000000,0.000000,0.839572,54,0,1470
9,task_9,1.000000,0.025316,0.049383,0.817948,79,2,1309


#### · Performancce on Test Set

In [89]:
trainer_mpnn.test(mpnn, test_loader)

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

Testing DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 24.45it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/roc          │    0.8635791540145874     │
└───────────────────────────┴───────────────────────────┘

[{'test/roc': 0.8635791540145874}]

In [90]:
pred_probs = trainer_mpnn.predict(mpnn, dataloaders=test_loader)
probs = torch.cat(pred_probs, dim=0).cpu().numpy()
all_y = []
for batch in test_loader:
    y = batch.Y  # 👈 get the label tensor from TrainingBatch
    all_y.append(y)
y_true = torch.cat(all_y, dim=0).cpu().numpy()  # shape: (num_samples, num_tasks)

model0_report_test = compute_classification_report(y_true, probs, threshold=0.5)
model0_report_test


/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 31.25it/s]
 - Precision: 0.5363
 - Recall: 0.2004
 - F1: 0.2710
 - AUC: 0.8313


,task,precision,recall,f1,auc,positives,predicted_positives,total_num
0,task_0,0.785714,0.343750,0.478261,0.687767,32,14,734
1,task_1,0.615385,0.444444,0.516129,0.855003,18,13,682
2,task_2,0.674419,0.376623,0.483333,0.909267,77,43,667
3,task_3,0.000000,0.000000,0.000000,0.844424,36,0,577
4,task_4,0.650000,0.160494,0.257426,0.762357,81,20,615
5,task_5,0.750000,0.176471,0.285714,0.859644,34,8,701
6,task_6,0.000000,0.000000,0.000000,0.857184,17,0,634
7,task_7,0.714286,0.217391,0.333333,0.804112,92,28,599
8,task_8,0.000000,0.000000,0.000000,0.816505,31,0,708
9,task_9,1.000000,0.057143,0.108108,0.846655,35,2,667


### 2) With Focal Loss

In [91]:
import pytorch_lightning as pl

In [92]:
alphas = []
for task in tox21_tasks:
    pos = tox21_data[task].sum()
    total = tox21_data[task].notna().sum()
    alpha = 1 - (pos / total)
    alphas.append(alpha)

# Create a tensor
alpha_tensor = torch.tensor(alphas, dtype=torch.float32)

In [98]:
mp = chemprop.nn.BondMessagePassing()
agg = chemprop.nn.MeanAggregation()
ffn = nn.Sequential(
    nn.Linear(300, 300),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Dropout(0.7),
    nn.Linear(300, 12)
    )

# ffn = chemprop.nn.BinaryClassificationFFN(n_tasks=len(tox21_tasks), dropout=0.7)
        
batch_norm = False
metric_list = None
mpnn_focal = MPNNModel_FocalLoss(mp, agg, ffn, batch_norm, metric_list, alpha_tensor, gamma=2.5)
mpnn_focal.load_state_dict(torch.load("../model/mpnn_focal_model.pt"))


<All keys matched successfully>

#### · Performancce on Validation Set

In [99]:
trainer_mpnn_focal = pl.Trainer()
trainer_mpnn_focal.validate(mpnn_focal, val_loader)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Validation DataLoader 0: 100%|██████████| 25/25 [00:01<00:00, 23.12it/s]Best thresholds: [np.float64(0.6473684210526316), np.float64(0.6473684210526316), np.float64(0.5210526315789473), np.float64(0.4789473684210527), np.float64(0.4789473684210527), np.float64(0.6052631578947368), np.float64(0.5631578947368421), np.float64(0.4789473684210527), np.float64(0.6052631578947368), np.float64(0.6052631578947368), np.float64(0.5210526315789473), np.float64(0.5210526315789473)]
Best thresholds: [np.float64(0.6473684210526316), np.float64(0.6473684210526316), np.float64(0.5210526315789473), np.float64(0.4789473684210527), np.float64(0.4789473684210527), np.float64(0.6052631578947368), np.float64(0.5631578947368421), np.float64(0.4789473684210527), np.float64(0.6052631578947368), np.float64(0.6052631578947368), np.float64(0.5210526315789473), np.float64(0.5210526315789473)]
Validation DataLoader 0: 100%|██████████| 25/25 [00:01<00:00, 20.40it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       val_mean_auc        │     0.827231764793396     │
└───────────────────────────┴───────────────────────────┘

[{'val_mean_auc': 0.827231764793396}]

In [100]:
preds = trainer_mpnn_focal.predict(mpnn_focal, dataloaders=val_loader)[:-1]

probs = torch.cat([r["probs"] for r in preds], dim=0).cpu().numpy()
y_true = torch.cat([r["targets"] for r in preds], dim=0).cpu().numpy()

model1_report_val = compute_classification_report(y_true, probs, threshold=0.55)
model1_report_val

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 26/26 [00:00<00:00, 31.39it/s]
 - Precision: 0.3918
 - Recall: 0.4523
 - F1: 0.3901
 - AUC: 0.8272


,task,precision,recall,f1,auc,positives,predicted_positives,total_num
0,task_0,0.283186,0.533333,0.369942,0.822213,60,113,1501
1,task_1,0.523810,0.550000,0.536585,0.840012,60,63,1400
2,task_2,0.638298,0.555556,0.594059,0.899022,162,141,1349
3,task_3,0.333333,0.142857,0.200000,0.843812,56,24,1200
4,task_4,0.595506,0.311765,0.409266,0.733977,170,89,1269
5,task_5,0.306122,0.562500,0.396476,0.796071,80,147,1429
6,task_6,0.243243,0.450000,0.315789,0.822043,40,74,1342
7,task_7,0.524476,0.394737,0.450450,0.820518,190,143,1184
8,task_8,0.162304,0.574074,0.253061,0.847575,54,191,1470
9,task_9,0.245033,0.468354,0.321739,0.800422,79,151,1309


#### · Performance on Test set

In [101]:
trainer_mpnn_focal.test(mpnn_focal, test_loader)

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 22.49it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_mean_auc       │    0.8391542434692383     │
│    test_recall_task_0     │          0.53125          │
│    test_recall_task_1     │    0.6666666865348816     │
│    test_recall_task_10    │    0.6666666865348816     │
│    test_recall_task_11    │    0.7894737124443054     │
│    test_recall_task_2     │    0.6623376607894897     │
│    test_recall_task_3     │    0.4444444477558136     │
│    test_recall_task_4     │     0.395061731338501     │
│    test_recall_task_5     │    0.8529411554336548     │
│    test_recall_task_6     │    0.5882353186607361     │
│    test_recall_task_7     │    0.6304348111152649     │
│    test_recall_task_8     │     0.774193525314331     │
│    test_recall_task_9     │     0.800000011920929     │
└───────────────────────────┴───────────────────────────┘

[{'test_recall_task_0': 0.53125,
  'test_recall_task_1': 0.6666666865348816,
  'test_recall_task_2': 0.6623376607894897,
  'test_recall_task_3': 0.4444444477558136,
  'test_recall_task_4': 0.395061731338501,
  'test_recall_task_5': 0.8529411554336548,
  'test_recall_task_6': 0.5882353186607361,
  'test_recall_task_7': 0.6304348111152649,
  'test_recall_task_8': 0.774193525314331,
  'test_recall_task_9': 0.800000011920929,
  'test_recall_task_10': 0.6666666865348816,
  'test_recall_task_11': 0.7894737124443054,
  'test_mean_auc': 0.8391542434692383}]

In [102]:
preds = trainer_mpnn_focal.predict(mpnn_focal, dataloaders=test_loader)

probs = torch.cat([r["probs"] for r in preds], dim=0).cpu().numpy()
y_true = torch.cat([r["targets"] for r in preds], dim=0).cpu().numpy()

model1_report_test = compute_classification_report(y_true, probs, threshold=0.55)
model1_report_test

/opt/anaconda3/envs/chemprop_env/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

Predicting DataLoader 0: 100%|██████████| 13/13 [00:00<00:00, 25.56it/s]
 - Precision: 0.3958
 - Recall: 0.4309
 - F1: 0.3742
 - AUC: 0.8392


,task,precision,recall,f1,auc,positives,predicted_positives,total_num
0,task_0,0.261905,0.343750,0.297297,0.702235,32,42,734
1,task_1,0.357143,0.555556,0.434783,0.896670,18,28,682
2,task_2,0.610169,0.467532,0.529412,0.910566,77,59,667
3,task_3,0.416667,0.138889,0.208333,0.830766,36,12,577
4,task_4,0.586207,0.209877,0.309091,0.737689,81,29,615
5,task_5,0.349206,0.647059,0.453608,0.888085,34,63,701
6,task_6,0.130435,0.176471,0.150000,0.821813,17,23,634
7,task_7,0.540984,0.358696,0.431373,0.808250,92,61,599
8,task_8,0.179775,0.516129,0.266667,0.849335,31,89,708
9,task_9,0.298701,0.657143,0.410714,0.861799,35,77,667
